In [0]:
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import from_json, col, current_timestamp

In [0]:
eh_conn_str = dbutils.secrets.get(
    "kv-scope",
    "eventhub-kafka-connection-string"
)

print("Secret retrieved successfully")
print("Secret length:", len(eh_conn_str))

Secret retrieved successfully
Secret length: 153


In [0]:
orders_schema = (
    StructType()
    .add("event_id", StringType())
    .add("order_id", StringType())
    .add("customer_id", StringType())
    .add("product_id", StringType())
    .add("quantity", IntegerType())
    .add("amount", DoubleType())
    .add("currency", StringType())
    .add("timestamp", StringType())
    .add("status", StringType())
)

In [0]:
eh_namespace = "evhns-nexpulse.servicebus.windows.net:9093"

kafka_sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{eh_conn_str}";'
)

In [0]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", eh_namespace)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", kafka_sasl_config)
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

print("Kafka/Event Hubs stream created successfully")

Kafka/Event Hubs stream created successfully


In [0]:
parsed = (
    raw_stream.select(
        from_json(
            col("value").cast("string"),
            orders_schema
        ).alias("data"),

        col("timestamp").alias("kafka_ingest_time"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
    )
    .select(
        "data.*",
        "kafka_ingest_time",
        "kafka_partition",
        "kafka_offset"
    )
    .withColumn(
        "bronze_loaded_at",
        current_timestamp()
    )
)

print("Orders JSON parsing configured successfully")

Orders JSON parsing configured successfully


In [0]:
bronze_path = (
    "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/orders/"
)

checkpoint_path = (
    "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/"
    "_checkpoints/orders/"
)

print("Bronze path:", bronze_path)
print("Checkpoint path:", checkpoint_path)

Bronze path: abfss://bronze@adlsnexpulse01.dfs.core.windows.net/orders/
Checkpoint path: abfss://bronze@adlsnexpulse01.dfs.core.windows.net/_checkpoints/orders/


In [0]:
query = (
    parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="30 seconds")
    .start(bronze_path)
)

print("Streaming query started")
print("Query ID:", query.id)
print("Status:", query.status)

query.awaitTermination()

Streaming query started
Query ID: f74461b4-b442-4c25-9f42-7d8f2670feea
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
